# Sistema integrado — XGBoost d12 + Magic UID + Count Encoding

Notebook **original de este proyecto** que integra las mejores técnicas de las soluciones
públicas de la competencia [IEEE-CIS Fraud Detection](https://www.kaggle.com/competitions/ieee-fraud-detection)
(Kaggle, 2019) — *inspirado* en ellas, no copiado:

| Técnica | Origen de la idea |
|---|---|
| Normalización de columnas D (`D − TransactionDT/86400`) | solución 1.er puesto |
| Magic UID `card1+addr1+floor(day−D1)` + agregaciones de grupo | solución 1.er puesto |
| Count/frequency encoding + interacciones `card1×card5`, `addr1×card1` | mejor LightGBM público |
| `fillna(-999)` / faltantes como categoría | baseline XGBoost simple |
| Validación temporal honesta (holdout 75/25 + GroupKFold por mes) | protocolo de la competencia |

**Semilla global: `SEED = 42`** (afecta a XGBoost: `subsample`, `colsample_bytree` y el
binning de GPU; el split GroupKFold por mes es determinístico).

In [1]:
import gc, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)
print("numpy", np.__version__, "| pandas", pd.__version__, "| xgboost", xgb.__version__, "| SEED =", SEED)

numpy 2.5.3 | pandas 3.0.6 | xgboost 3.4.1 | SEED = 42


## 1. Carga de datos

Solo `train` (590 540 transacciones): la validación es local y temporal, no hay submission.

In [2]:
%%time
str_type = ['ProductCD','card4','card6','P_emaildomain','R_emaildomain','M1','M2','M3','M4','M5',
            'M6','M7','M8','M9','id_12','id_15','id_16','id_23','id_27','id_28','id_29','id_30',
            'id_31','id_33','id_34','id_35','id_36','id_37','id_38','DeviceType','DeviceInfo']
cols = ['TransactionID','TransactionDT','TransactionAmt','ProductCD','card1','card2','card3',
        'card4','card5','card6','addr1','addr2','dist1','dist2','P_emaildomain','R_emaildomain',
        'C1','C2','C3','C4','C5','C6','C7','C8','C9','C10','C11','C12','C13','C14',
        'D1','D2','D3','D4','D5','D6','D7','D8','D9','D10','D11','D12','D13','D14','D15',
        'M1','M2','M3','M4','M5','M6','M7','M8','M9']
v = [1,3,4,6,8,11] + [13,14,17,20,23,26,27,30] + [36,37,40,41,44,47,48] + [54,56,59,62,65,67,68,70]
v += [76,78,80,82,86,88,89,91] + [107,108,111,115,117,120,121,123] + [124,127,129,130,136]
v += [138,139,142,147,156,162,165,160,166] + [178,176,173,182] + [187,203,205,207,215]
v += [169,171,175,180,185,188,198,210,209] + [218,223,224,226,228,229,235] + [240,258,257,253,252,260,261]
v += [264,266,267,274,277] + [220,221,234,238,250,271] + [294,284,285,286,291,297]
v += [303,305,307,309,310,320] + [281,283,289,296,301,314]
cols += ['V'+str(x) for x in v]

dtypes = {c: 'float32' for c in cols + ['id_0'+str(i) for i in range(1,10)] + ['id_'+str(i) for i in range(10,34)]}
for c in str_type: dtypes[c] = 'category'

X = pd.read_csv('data/raw/train_transaction.csv', index_col='TransactionID', dtype=dtypes, usecols=cols+['isFraud'])
tid = pd.read_csv('data/raw/train_identity.csv', index_col='TransactionID', dtype=dtypes)
X = X.merge(tid, how='left', left_index=True, right_index=True)
del tid
y = X['isFraud'].copy()
del X['isFraud']
gc.collect()
print('Train:', X.shape, '| fraude:', f'{y.mean():.4%}')

Train: (590540, 213) | fraude: 3.4990%
CPU times: user 5.48 s, sys: 477 ms, total: 5.96 s
Wall time: 6 s


## 2. Normalización de columnas D

Las columnas D son *deltas* respecto a un momento del pasado y crecen con el tiempo
(deriva temporal). Se convierten en fechas absolutas: `D15n = TransactionDT/(24·3600) − D15`,
lo que las hace transferibles al futuro.

In [3]:
%%time
for i in range(1, 16):
    if i in [1, 2, 3, 5, 9]:   # D1-D3, D5, D9 ya son estables
        continue
    X['D'+str(i)] = X['D'+str(i)] - X.TransactionDT / np.float32(24*60*60)
print('D columns normalizadas')

D columns normalizadas
CPU times: user 6 ms, sys: 8.95 ms, total: 15 ms
Wall time: 14.9 ms


## 3. Codificación de categóricas y faltantes

Categóricas → códigos enteros (`factorize`); numéricas se desplazan a positivo y los
`NaN` se marcan con `-1` (los GBM tratan los faltantes como categoría informativa).

In [4]:
%%time
for f in X.columns:
    if X[f].dtype.name in ('category', 'object', 'str'):
        codes, _ = pd.factorize(X[f], sort=True)
        X[f] = codes.astype('int16')
    elif f not in ['TransactionAmt', 'TransactionDT']:
        X[f] = X[f] - np.float32(X[f].min())
        X[f] = X[f].fillna(-1)
print('categóricas codificadas | NaN -> -1')

categóricas codificadas | NaN -> -1
CPU times: user 565 ms, sys: 175 ms, total: 740 ms
Wall time: 744 ms


## 4. Count encoding, interacciones y señales de monto

Frecuencia de tarjeta/ dirección como señal de riesgo; entidades más finas
`card1×card5` y `addr1×card1`; centavos del monto y features FE.

In [5]:
%%time
def encode_FE(df, cols):
    for col in cols:
        vc = df[col].value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        df[col+'_FE'] = df[col].map(vc).astype('float32')

def encode_CB(col1, col2, df):
    nm = col1+'_'+col2
    df[nm] = df[col1].astype(str)+'_'+df[col2].astype(str)
    codes, _ = pd.factorize(df[nm], sort=True)
    df[nm] = codes.astype('int32')

# centavos del monto (señal de fraude en montos redondos)
X['cents'] = (X['TransactionAmt'] - np.floor(X['TransactionAmt'])).astype('float32')

# count/frequency encoding sobre entidades base
encode_FE(X, ['addr1','card1','card2','card3','P_emaildomain'])

# interacciones como entidades más finas (card1×card5, addr1×card1)
encode_CB('card1','card5', X)
encode_CB('addr1','card1', X)
encode_FE(X, ['card1_card5','addr1_card1'])
print('FE + interacciones listas:', X.shape[1], 'columnas')

<timed exec>:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert

<timed exec>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


FE + interacciones listas: 223 columnas
CPU times: user 1.02 s, sys: 104 ms, total: 1.13 s
Wall time: 1.13 s


<timed exec>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


## 5. Magic UID + agregaciones de grupo

`UID = card1_addr1 + floor(day − D1)` reconstruye el pseudo-cliente. Se agregan su
frecuencia, media y std de montos/Ds/Cs/Ms y nunique de contactos. El UID **se elimina
antes de entrenar**; solo sirve como llave de agregación.

In [6]:
%%time
def encode_AG(main_columns, uids, aggregations=['mean'], df=None, usena=False):
    df = X if df is None else df
    for main_column in main_columns:
        for col in uids:
            for agg_type in aggregations:
                nm = main_column+'_'+col+'_'+agg_type
                temp = df[[col, main_column]].copy()
                if usena:
                    temp.loc[temp[main_column] == -1, main_column] = np.nan
                agg = temp.groupby(col)[main_column].agg([agg_type])
                df[nm] = df[col].map(agg[agg_type]).astype('float32')
                df[nm] = df[nm].fillna(-1)

def encode_AG2(main_columns, uids, df=None):
    df = X if df is None else df
    for main_column in main_columns:
        for col in uids:
            mp = df.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            df[col+'_'+main_column+'_ct'] = df[col].map(mp).astype('float32')

X['day'] = X.TransactionDT / (24*60*60)
START_DATE = np.datetime64('2017-11-30T00:00:00')
dt = pd.to_datetime(START_DATE) + pd.to_timedelta(X['TransactionDT'], unit='s')
X['DT_M'] = (dt.dt.year - 2017) * 12 + dt.dt.month
X['uid'] = X.addr1_card1.astype(str) + '_' + np.floor(X.day - X.D1).astype(str)
encode_FE(X, ['uid'])
encode_AG(['TransactionAmt','D4','D9','D10','D15'], ['uid'], ['mean','std'], usena=True)
encode_AG(['C'+str(x) for x in range(1,15) if x != 3], ['uid'], ['mean'], usena=True)
encode_AG(['M'+str(x) for x in range(1,10)], ['uid'], ['mean'], usena=True)
encode_AG2(['P_emaildomain','dist1','DT_M','id_02','cents'], ['uid'])
encode_AG(['C14'], ['uid'], ['std'], usena=True)
encode_AG2(['C13','V314'], ['uid'])
encode_AG2(['V127','V136','V309','V307','V320'], ['uid'])
X['outsider15'] = (np.abs(X.D1 - X.D15) > 3).astype('int8')
print('magic UID + agregaciones listas:', X.shape[1], 'columnas')

<timed exec>:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<timed exec>:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<timed exec>:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<timed exec>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<timed exec>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


...

<timed exec>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<timed exec>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


<timed exec>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


magic UID + agregaciones listas: 273 columnas
CPU times: user 17.7 s, sys: 135 ms, total: 17.8 s
Wall time: 18 s


<timed exec>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<timed exec>:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


## 6. Validación temporal honesta

Dos protocolos, ambos hacia el futuro: **holdout temporal 75/25** y **GroupKFold
mensual ×3** (oof). Se fija `SEED` en el modelo; el split por meses es determinístico.

In [7]:
%%time
cols_magic = [c for c in X.columns if c not in ('TransactionDT','DT_M','day','uid')]
for c in ['D6','D7','D8','D9','D12','D13','D14','C3','M5','id_08','id_33','card4',
          'id_07','id_14','id_21','id_30','id_32','id_34'] + ['id_'+str(x) for x in range(22,28)]:
    if c in cols_magic: cols_magic.remove(c)

# ablation: base = magic SIN las features derivadas del UID
cols_base = [c for c in cols_magic if 'uid' not in c and c != 'outsider15']
print('features sin UID:', len(cols_base), '| con magic UID:', len(cols_magic))

def fit_eval(feature_cols, tag):
    res = {}
    params = dict(n_estimators=2000, max_depth=12, learning_rate=0.02,
                  subsample=0.8, colsample_bytree=0.4, missing=-1,
                  eval_metric='auc', early_stopping_rounds=100,
                  tree_method='hist', device='cuda', random_state=SEED)
    t0 = time.time()
    cut = 3 * len(X) // 4
    idxT, idxV = X.index[:cut], X.index[cut:]
    clf = xgb.XGBClassifier(**params)
    clf.fit(X.loc[idxT, feature_cols], y[idxT],
            eval_set=[(X.loc[idxV, feature_cols], y[idxV])], verbose=500)
    res['holdout'] = roc_auc_score(y[idxV], clf.predict_proba(X.loc[idxV, feature_cols])[:,1])
    del clf; gc.collect()

    oof = np.zeros(len(X))
    gk = GroupKFold(n_splits=3)
    for i, (tr, va) in enumerate(gk.split(X, y, groups=X['DT_M'])):
        clf = xgb.XGBClassifier(**{**params, 'n_estimators': 5000, 'early_stopping_rounds': 200})
        clf.fit(X[feature_cols].iloc[tr], y.iloc[tr],
                eval_set=[(X[feature_cols].iloc[va], y.iloc[va])], verbose=0)
        oof[va] = clf.predict_proba(X[feature_cols].iloc[va])[:,1]
        del clf; gc.collect()
        print(f'[{tag}] fold {i+1}/3 listo')
    res['oof'] = roc_auc_score(y, oof)
    res['s'] = time.time() - t0
    print(f"[{tag}] holdout={res['holdout']:.4f} | OOF GroupKFold x3={res['oof']:.4f} | {res['s']:.0f}s")
    return res

features sin UID: 198 | con magic UID: 245
CPU times: user 175 μs, sys: 0 ns, total: 175 μs
Wall time: 170 μs


## 7. Ablación: sin UID vs. con magic UID

Mismo algoritmo, mismo hardware, misma semilla — cambia solo la representación de datos.

In [8]:
%%time
r_base = fit_eval(cols_base, 'sin UID  ')
r_magic = fit_eval(cols_magic, 'con magic')

print()
print('=' * 62)
print(f"{'modelo':<28}{'holdout':>10}{'OOF x3':>10}{'tiempo':>10}")
print('-' * 62)
print(f"{'XGBoost d12 sin UID':<28}{r_base['holdout']:>10.4f}{r_base['oof']:>10.4f}{r_base['s']:>8.0f}s")
print(f"{'XGBoost d12 + magic UID':<28}{r_magic['holdout']:>10.4f}{r_magic['oof']:>10.4f}{r_magic['s']:>8.0f}s")
print('-' * 62)
print(f"{'GANANCIA (representación)':<28}{r_magic['holdout']-r_base['holdout']:>10.4f}{r_magic['oof']-r_base['oof']:>10.4f}")
print('=' * 62)

[0]	validation_0-auc:0.81110


[500]	validation_0-auc:0.93335


[816]	validation_0-auc:0.93376


/home/edgarchambilla/UTEC/PROYECTO1_PLANIFICA/.venv/lib/python3.12/site-packages/xgboost/core.py:569: UserWarning: [00:37:22] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[sin UID  ] fold 1/3 listo


...

[500]	validation_0-auc:0.94551


[596]	validation_0-auc:0.94493


[con magic] fold 1/3 listo


[con magic] fold 2/3 listo


[con magic] fold 3/3 listo
[con magic] holdout=0.9456 | OOF GroupKFold x3=0.9509 | 168s

modelo                         holdout    OOF x3    tiempo
--------------------------------------------------------------
XGBoost d12 sin UID             0.9340    0.9347     155s
XGBoost d12 + magic UID         0.9456    0.9509     168s
--------------------------------------------------------------
GANANCIA (representación)       0.0116    0.0162
CPU times: user 7min 28s, sys: 6.81 s, total: 7min 35s
Wall time: 5min 22s


## 8. Conclusión

La ganancia proviene de la **representación de datos** (magic UID + agregaciones),
no del algoritmo ni del hardware: con el mismo XGBoost d12 y la misma semilla
(`SEED = 42`), reconstruir el pseudo-cliente y agregar su historial mejora el AUC
local en ~0.01. La validación debe ser temporal (holdout 75/25 u OOF mensual);
un KFold aleatorio inflaría el resultado.